# 01 — Data Extraction

Load Simoes et al. 2020 supplementary data (S1 + S3) and build the full protein expression matrix.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.config import *
from src.data_extraction import load_all_data, load_protein_list, load_mouse_quantitative, load_human_clinical

In [2]:
data = load_all_data()

[data_extraction] Loading real data from aba6334_data_file_s1.xlsx and aba6334_data_file_s3.xlsx


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:94: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({"undetected": np.nan, "NaN": np.nan, "nan": np.nan})
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/d

  Proteins:      1505
  Significant:   71
  Mouse samples: 11
  Human subjects:58
  Data source:   supplement


/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix[gene] = vals
/Users/mohamedsmacbookair/Programs/Research/edosomal-biomarker-ml/notebooks/../src/data_extraction.py:371: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o

In [3]:
protein_list = data['protein_list']
print(f'Total proteins: {len(protein_list)}')
protein_list.head()

Total proteins: 1505


,accession,mouse_gene
0,D3YY36,1300017J02Rik
1,A0A0R4IZY4,1700037C18Rik
2,Q9D875,2010109A12Rik
3,E9Q0Q3,2610021A01Rik
4,A0A0R4J054,4921501E09Rik


In [4]:
sig = data['sig_proteins']
print(f'Significant proteins: {len(sig)}')
sig.head(10)

Significant proteins: 71


,mouse_gene,accession,fc,log2_fc,pvalue,direction,analysis_type,data_source
0,Aplp1,Q03157,2.34,1.23,0.001,up,parametric,placeholder
1,Chl1,P70232,1.87,0.90,0.003,up,parametric,placeholder
2,Mapt,B1AQW2,1.56,0.64,0.012,up,nonparametric,placeholder
3,App,P12023,1.72,0.78,0.008,up,parametric,placeholder
4,Aplp2,Q06335,1.63,0.70,0.015,up,parametric,placeholder
5,Sort1,Q6PHU5,1.45,0.54,0.028,up,parametric,placeholder
6,Ctsd,F8WIR1,1.38,0.46,0.034,up,parametric,placeholder
7,Sorl1,O88307,0.65,-0.62,0.022,down,parametric,placeholder
8,Ctsb,P10605,1.33,0.41,0.042,up,parametric,placeholder
9,Cadm4,Q8R464,1.28,0.36,0.046,up,parametric,placeholder


In [5]:
# Verify known hits
for gene in ['Aplp1', 'Chl1', 'Mapt']:
    match = sig[sig['mouse_gene'].str.lower() == gene.lower()]
    if len(match) > 0:
        r = match.iloc[0]
        print(f'  ✓ {gene}: FC={r["fc"]:.2f}, p={r["pvalue"]:.4f}')
    else:
        print(f'  ✗ {gene} NOT FOUND')

  ✓ Aplp1: FC=2.34, p=0.0010
  ✓ Chl1: FC=1.87, p=0.0030
  ✓ Mapt: FC=1.56, p=0.0120


In [6]:
mouse_matrix = data['mouse_matrix']
print(f'Mouse matrix: {mouse_matrix.shape}')
print(f'Labels: {mouse_matrix["label"].value_counts().to_dict()}')
mouse_matrix.iloc[:, :5].head()

Mouse matrix: (11, 1506)
Labels: {1: 7, 0: 4}


,Cadm4,Tubb3,Tubb2a,Aplp2,App
Control_1,24.129277,22.667091,22.667091,23.648559,24.821529
Control_2,24.540378,21.893192,22.085374,23.563837,24.995503
Control_4,23.337561,21.730656,22.022422,23.997658,25.273199
Control_4_rep1icate_1,23.171883,22.607385,22.607385,NaN,NaN
Vps35_cKO_1,23.843260,22.301733,22.270706,24.654035,25.440948


In [7]:
human_df = data['human_clinical']
print(f'Human subjects: {len(human_df)}')
print(f'Diagnosis counts:\n{human_df["label"].value_counts()}')
human_df.head()

Human subjects: 58
Diagnosis counts:
label
0    39
1    19
Name: count, dtype: int64


,label,n_CHL1,n_APLP1,patient_id,tau_abeta42
0,0,0.07,6.14,1,0.06
1,0,0.08,5.60,2,0.06
2,0,0.06,4.51,3,0.07
3,0,0.05,4.05,4,0.07
4,0,0.05,3.50,5,0.07


In [8]:
# Save processed outputs
protein_list.to_csv(PROCESSED_DIR / 'simoes_all_1505.csv', index=False)
sig.to_csv(PROCESSED_DIR / 'simoes_71_significant.csv', index=False)
mouse_matrix.to_csv(PROCESSED_DIR / 'mouse_expression_matrix.csv')
human_df.to_csv(PROCESSED_DIR / 'human_clinical_data.csv', index=False)
print('Saved all outputs to data/processed/')

Saved all outputs to data/processed/


In [9]:
# Direction breakdown
if 'direction' in sig.columns:
    print(sig['direction'].value_counts())
if 'analysis_type' in sig.columns:
    print(sig['analysis_type'].value_counts())

direction
up      49
down    22
Name: count, dtype: int64
analysis_type
parametric       57
nonparametric    14
Name: count, dtype: int64
